#  Fase 1 - Exploracion de encuesta de movilidad

## Indice

- **Conceptos y terminos importantes**
- **Exploracion y caracterizacion Encuesta de Movilidad Bogotá 2023 (EODH)**
- **Datasets y archivos esenciales**
- **Exploracion y caracterizacion de la Encuesta de Movilidad de Bogotá 2023 para dataset (EODH)**
    - **Exploracion de metadatos, y ejemplo de valores**
    - **Exploracion de valores categoricos o estaticos o repetidos**

## Conceptos y terminos importantes

- **EODH(Encuesta-Origen-Destino-Hogares):** Son las encuestas de puntos de origen y puntos de llegada realizados en los hogares de las personas encuestadas


## Datasets y archivos esenciales

> **Nota:** los nombres que estan encerrados en **' '** no hacen parte del nombre, solo se agregan porque el nombre del archivo o directorio contiene espacios

**Aclaracion:** Todos los archivos que se listan y muestran en esta seccion se asumen que tienen como directorio padre, el directorio generando al correr el comando `unzip <nombre_dataset_encuesta_movilidad_bogota_2023>.zip`. Es decir al correr el comando anterior se genera el directorio **2.PublicacionSIMUR/** para este dataset 2023 en especifico. Es a partir de este directorio raiz que se tomara como punto de partida para listar los demas archivos. Los archivos relevantes para el analisis estan bajo **2.PublicacionSIMUR/EODH/'05_Base datos procesada'/CSV/**

**Archivos:**
- 'a. Modulo hogares.csv'
- 'b. Modulo vehiculos.csv'
- 'c. Modulo personas.csv'
- 'd. Modulo viajes.csv'
- 'e. Modulo etapas.csv'

## Dependencias, librerias y funciones helpers que facilitan el analisis y exploracion de los datasets

In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("CSV") # tenemos acceso a los archivos listados en la seccion "Datasets y archivos esenciales"

def cargar_csv(nombre):
    ruta = BASE_DIR / nombre

    try:
        df = pd.read_csv(ruta, encoding="utf-8", sep=None, engine="python")
    except UnicodeDecodeError:
        df = pd.read_csv(ruta, encoding="latin-1", sep=None, engine="python")
    
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    return df

# cargamos los datasets

hogares = cargar_csv("a. Modulo hogares.csv")
vehiculos = cargar_csv("b. Modulo vehiculos.csv")
personas = cargar_csv("c. Modulo personas.csv")
viajes = cargar_csv("d. Modulo viajes.csv")
etapas = cargar_csv("e. Modulo etapas.csv")



In [24]:
# funciones para explorar informacion

def explorar_metadata_data_set(df):
    print(f"Filas: {df.shape[0]}")
    print(f"Columnas:")
    for col in df.columns:
        print(f"- {col}")

def explorar_data_set(df, n=3):
    print(f"Nulos: {df.isna().sum().sum()}")
    print(f"Duplicados: {df.duplicated().sum()}\n")

    print(f"Valores de los {n} primeros registros")
    print(f"ATENCION: formato del output a continuacion: <nombre_columna>: <val1>, <val2>, ..., <val{n}>")
    print(f"val1, val2, ..., val{n} corresponden al los valores de los registros para la columna en especifico\n")
    
    headN = df.head(n)
    for columna, valores in zip(headN.columns, headN.T.values):
        print(f"{columna}: {', '.join(map(str, valores))}")

def explorar_categorias(
    df: pd.DataFrame,
    max_valores: int = 30,
    ignorar_prefijos=("key_", "cod_", "fexp")
):
    """
    Muestra columnas categóricas con un número reducido de valores únicos.
    """

    for columna in df.columns:
        nombre = columna.lower()
        if nombre.startswith(ignorar_prefijos): continue

        if not (
            pd.api.types.is_object_dtype(df[columna])
            or pd.api.types.is_string_dtype(df[columna])
            or isinstance(df[columna], pd.CategoricalDtype)
        ):
            continue
 
        valores = (
            df[columna]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
        )

        if len(valores) <= 1 or len(valores) > max_valores: continue

        print(f"\n{columna} ({len(valores)} valores)")
        print("-" * len(f"{columna} ({len(valores)} valores)"))

        for valor in sorted(valores):
            print(f"• {valor}")

def explorar_keys(nombre_df: str, df: pd.DataFrame, datasets: dict):
    print(f"=== {nombre_df} ===")

    # Posibles Primary Keys
    pks = [col for col in df.columns if df[col].is_unique]

    if not pks:
        print("\nPrimary Keys: No se encontraron.")
        return

    print("\nPrimary Keys:")
    for pk in pks:
        ejemplos = "\n\t\t- ".join(map(str, df[pk].head(3).tolist()))
        
        print(f"• {pk}")
        print(f"\tEjemplos:\n\t\t- {ejemplos}")

        referencias = []
        for otro_nombre, otro_df in datasets.items():
            
            if otro_nombre == nombre_df: continue
            if pk in otro_df.columns: referencias.append(otro_nombre)
                
        if referencias: print(f"    Referenciada en: {', '.join(referencias)}")
        else: print("    Referenciada en: Ningún dataset")

    print("\nForeign Keys:")

    for columna in df.columns:

        if columna in pks: continue

        referencias = []

        for otro_nombre, otro_df in datasets.items():

            if otro_nombre == nombre_df: continue
            if columna in otro_df.columns: referencias.append(otro_nombre)

        if referencias: print(f"• {columna} -> {', '.join(referencias)}")

## Exploracion y caracterizacion de la Encuesta de Movilidad de Bogotá 2023 para dataset (EODH)

### Exploracion de metadatos, y ejemplo de valores

#### Hogares

In [3]:
print("Dataset: Hogares")
explorar_metadata_data_set(hogares)
explorar_data_set(hogares)

Dataset: Hogares
Filas: 22755
Columnas:
- fecha
- cod_viv
- cod_hog
- cod_utam_hg
- nom_utam_hg
- zat_hg
- cod_upl_hg
- nom_upl_hg
- cod_mpio_hg
- nom_mpio_hg
- cod_loc_hg
- nom_loc_hg
- estrato_hg
- cod_dane_manzana_hg
- tipo_enc
- tipo_zona_hg
- cod_barrio_vereda_hg
- nom_barrio_vereda_hg
- tipo_viv
- cant_hg_viv
- perstotal_hg
- pers>5años_hg
- pers>18años_hg
- ingre_mes_hg
- cómo_enteró
- key_hg
- fexp_hg
Nulos: 0
Duplicados: 0

Valores de los 3 primeros registros
ATENCION: formato del output a continuacion: <nombre_columna>: <val1>, <val2>, ..., <val3>
val1, val2, ..., val3 corresponden al los valores de los registros para la columna en especifico

fecha: 15/06/2023 18:05, 3/05/2023 18:25, 4/05/2023 17:05
cod_viv: 212, 213, 214
cod_hog: 20301, 6088, 13017
cod_utam_hg: UTAM035, UTAM035, UTAM035
nom_utam_hg: CIUDAD JARDIN, CIUDAD JARDIN, CIUDAD JARDIN
zat_hg: 450, 450, 450
cod_upl_hg: 16, 16, 16
nom_upl_hg: Restrepo, Restrepo, Restrepo
cod_mpio_hg: 11001, 11001, 11001
nom_mpio_hg: B

#### Vehiculos

In [4]:
print("Dataset: Vehiculos")
explorar_metadata_data_set(vehiculos)
explorar_data_set(vehiculos)

Dataset: Vehiculos
Filas: 17392
Columnas:
- cod_vh
- cod_viv
- cod_hog
- cod_utam_hg
- zat_hg
- cod_upl_hg
- nom_mun_hg
- estrato_hg
- tipo_vehículo
- cantidad
- tipo_combustible
- mun_matricula
- tipo_placa
- exento_pyp
- modelo
- propiedad_vh
- key_hg
- fexp_vh
Nulos: 0
Duplicados: 0

Valores de los 3 primeros registros
ATENCION: formato del output a continuacion: <nombre_columna>: <val1>, <val2>, ..., <val3>
val1, val2, ..., val3 corresponden al los valores de los registros para la columna en especifico

cod_vh: 1, 2, 3
cod_viv: 12156, 3274, 3274
cod_hog: 5102, 2130, 2130
cod_utam_hg: UTAM076, UTAM056, UTAM056
zat_hg: 309, 703, 703
cod_upl_hg: 10, 29, 29
nom_mun_hg: Bogotá D.C., Bogotá D.C., Bogotá D.C.
estrato_hg: 2, 2, 2
tipo_vehículo: Automóvil, Motocicleta, Motocicleta
cantidad: 1, 1, 1
tipo_combustible: Sólo Gasolina, Sólo Gasolina, Sólo Gasolina
mun_matricula: 66001, No aplica, No aplica
tipo_placa: Privada, Privada, Privada
exento_pyp: No, NA (motos), NA (motos)
modelo: 2008,

#### Personas

In [5]:
print("Dataset: Personas")
explorar_metadata_data_set(personas)
explorar_data_set(personas)

Dataset: Personas
Filas: 67556
Columnas:
- cod_per
- cod_hg
- nom_mun_hg
- cod_upl_hg
- cod_utam_hg
- zat_hg
- estra_hg
- orden
- edad
- sexo
- genero
- orien_sexual
- identidad_etnica
- madre_cab_familia
- max_nivel_edu
- ocupacion_principal
- actividad_economica
- permanencia_entre_semana
- quien_lleva/recoge_establecimiento
- cuidado_despues_regresar
- condicion_discapacidad
- RLCPD
- cuidado_entre_semana_discap
- dific_princ_medios_transp_discap
- licencia_cond_vigente
- posee_celular
- realiza_desplazamientos
- modo_principal_pre-pandemia
- modo_principal_pandemia
- razón_cambio_pandemia
- cambio_frecuencia_post-pandemia
- cambio_modo_pico_placa
- cambio_modo_grandes_obra
- cambio_modo_seguridad
- acto_violencia_sexual
- lugar_violencia_sexual
- a_quien_acudio
- key_hg
- key_persona
- fexp_per>5años
Nulos: 0
Duplicados: 0

Valores de los 3 primeros registros
ATENCION: formato del output a continuacion: <nombre_columna>: <val1>, <val2>, ..., <val3>
val1, val2, ..., val3 corresponde

#### Viajes

In [6]:
print("Dataset: Viajes")
explorar_metadata_data_set(viajes)
explorar_data_set(viajes)

Dataset: Viajes
Filas: 100174
Columnas:
- cod_hg
- nom_mun_hg
- cod_upl_hg
- cod_utam_hg
- zat_hg
- estra_hg
- cod_pers
- cod_vj
- orden_vj
- otro_vj
- zat_ori
- utam_ori
- upl_ori
- localidad_ori
- nom_mun_ori
- zat_des
- utam_des
- upl_des
- localidad_des
- nom_mun_des
- hora_ini
- hora_fin
- duracion_min
- t_acceso_min
- t_espera_min
- t_egreso_min
- modo_principal_agrupado
- modo_principal_desagrupado
- motivo_viaje
- motivo_viaje_cuidado
- frecuencia_viaje
- etapas
- app_antes_vj
- app_durante_vj
- edad
- sexo
- genero
- orien_sexual
- identidad_etnica
- madre_cab_familia
- max_nivel_edu
- ocupacion_principal
- key_hg
- key_pers
- key_pers_viaja
- key_viaje
- fexp_vj
Nulos: 0
Duplicados: 0

Valores de los 3 primeros registros
ATENCION: formato del output a continuacion: <nombre_columna>: <val1>, <val2>, ..., <val3>
val1, val2, ..., val3 corresponden al los valores de los registros para la columna en especifico

cod_hg: 1, 1, 1
nom_mun_hg: Bogotá, Bogotá, Bogotá
cod_upl_hg: 31, 31,

#### Etapas

In [7]:
print("Dataset: Etapas")
explorar_metadata_data_set(etapas)
explorar_data_set(etapas)

Dataset: Etapas
Filas: 110881
Columnas:
- cod_eta
- cod_vj
- cod_pers
- cod_hg
- nom_mun_hg
- cod_upl_hg
- cod_utam_hg
- zat_hg
- estra_hg
- orden_eta
- etapas
- modo_etapa
- modo_principal_vj
- uso_veh_hg_cod
- t_acceso_min
- t_espera_min
- pago_viaje
- costo_viaje
- modalidad_pago_vj
- lugar_parqueo
- t_parqueo_min
- costo_parqueo
- t_egreso_min
- motivo_viaje
- edad
- sexo
- genero
- orien_sexual
- identidad_etnica
- mad_cab_familia
- max_nivel_educativo
- ocupación_principal
- actividad_economica
- condicion_discapacidad
- licencia_vigente
- key_viaje
- key_etapa
- fexp_eta
Nulos: 0
Duplicados: 0

Valores de los 3 primeros registros
ATENCION: formato del output a continuacion: <nombre_columna>: <val1>, <val2>, ..., <val3>
val1, val2, ..., val3 corresponden al los valores de los registros para la columna en especifico

cod_eta: 1, 2, 3
cod_vj: 1, 2, 3
cod_pers: 13967, 13967, 13968
cod_hg: 1, 1, 1
nom_mun_hg: Bogotá, Bogotá, Bogotá
cod_upl_hg: 31, 31, 31
cod_utam_hg: UTAM043, UTAM043

### Exploracion de valores categoricos o estaticos o repetidos

Esta exploracion tiene el fin u objetivo de encontrar valores cualitativos constantes usados para las encuestas o valores categoricos, por ejemplo estratos, localidades, genero, orientacion sexual, etc.

#### Hogares

In [8]:
print("Dataset: Hogares")
explorar_categorias(hogares)

Dataset: Hogares

nom_mpio_hg (21 valores)
------------------------
• Bogotá D.C.
• Bojacá
• Cajicá
• Choachí
• Chía
• Cota
• El Rosal
• Facatativá
• Funza
• Gachancipá
• La Calera
• Madrid
• Mosquera
• Sibaté
• Soacha
• Sopó
• Subachoque
• Tabio
• Tenjo
• Tocancipá
• Zipaquirá

nom_loc_hg (21 valores)
-----------------------
• Antonio Nariño
• Barrios Unidos
• Bosa
• Chapinero
• Ciudad Bolívar
• Engativá
• Fontibón
• Kennedy
• La Candelaria
• Los Mártires
• No aplica
• Puente Aranda
• Rafael Uribe Uribe
• San Cristóbal
• Santa Fe
• Suba
• Sumapaz
• Teusaquillo
• Tunjuelito
• Usaquén
• Usme

estrato_hg (7 valores)
----------------------
• 1
• 2
• 3
• 4
• 5
• 6
• No aplica

tipo_enc (2 valores)
--------------------
• Muestra
• Reemplazo

tipo_zona_hg (2 valores)
------------------------
• Centro poblado
• Urbano

tipo_viv (5 valores)
--------------------
• Apartamento
• Casa
• Cuarto inquilinato
• Cuarto otro
• Otro tipo

ingre_mes_hg (12 valores)
-------------------------
• 0-400000
• 

#### Vehiculos

In [9]:
print("Dataset: Vehiculos")
explorar_categorias(vehiculos)

Dataset: Vehiculos

nom_mun_hg (21 valores)
-----------------------
• Bogotá D.C.
• Bojacá
• Cajicá
• Choachí
• Chía
• Cota
• El Rosal
• Facatativá
• Funza
• Gachancipá
• La Calera
• Madrid
• Mosquera
• Sibaté
• Soacha
• Sopó
• Subachoque
• Tabio
• Tenjo
• Tocancipá
• Zipaquirá

estrato_hg (7 valores)
----------------------
• 1
• 2
• 3
• 4
• 5
• 6
• No aplica

tipo_vehículo (21 valores)
--------------------------
• Automóvil
• Automóvil o camioneta de servicio especial
• Bicicleta motor eléctrico
• Bicicleta motor gasolina
• Bicicleta niños
• Bicicleta sin motor
• Bicitaxi motor eléctrico
• Bicitaxi motor gasolina
• Bicitaxi sin motor
• Camión
• Campero/Camioneta
• Moto - carro
• Motocicleta
• Otro
• Patineta
• Patineta eléctrica
• Pick Up/Van
• Taxi
• Triciclo - moto
• Vehículo tracción animal
• Vehículo tracción humana

tipo_combustible (9 valores)
----------------------------
• Diésel
• Eléctrico
• GNV
• GNV y gasolina
• Híbrido (eléctrico- gasolina / diesel)
• NS/NR
• No aplica
• O

#### Personas

In [10]:
print("Dataset: Personas")
explorar_categorias(personas)

Dataset: Personas

nom_mun_hg (21 valores)
-----------------------
• Bogotá
• Bojacá
• Cajicá
• Choachí
• Chía
• Cota
• El Rosal
• Facatativá
• Funza
• Gachancipá
• La Calera
• Madrid
• Mosquera
• Sibaté
• Soacha
• Sopó
• Subachoque
• Tabio
• Tenjo
• Tocancipá
• Zipaquirá

estra_hg (7 valores)
--------------------
• 1
• 2
• 3
• 4
• 5
• 6
• No aplica

sexo (3 valores)
----------------
• Hombre
• Intersexual
• Mujer

genero (6 valores)
------------------
• Femenino
• Masculino
• NS/NR
• No aplica
• No binario
• Transgénero

orien_sexual (5 valores)
------------------------
• Bisexual
• Heterosexual
• Homosexual
• NS/NR
• No aplica

identidad_etnica (6 valores)
----------------------------
• Gitana o Rrom
• Indígena
• Negra, mulata, afrodes.
• Ninguno
• Palenquero
• Raizal

madre_cab_familia (3 valores)
-----------------------------
• No
• No aplica
• Sí

max_nivel_edu (15 valores)
--------------------------
• Media completa (10° y 11°)
• Media incompleta (10° y 11°)
• Ninguno
• No aplica

#### Viajes

In [11]:
print("Dataset: Viajes")
explorar_categorias(viajes)

Dataset: Viajes

nom_mun_hg (21 valores)
-----------------------
• Bogotá
• Bojacá
• Cajicá
• Choachí
• Chía
• Cota
• El Rosal
• Facatativá
• Funza
• Gachancipá
• La Calera
• Madrid
• Mosquera
• Sibaté
• Soacha
• Sopó
• Subachoque
• Tabio
• Tenjo
• Tocancipá
• Zipaquirá

estra_hg (7 valores)
--------------------
• 1
• 2
• 3
• 4
• 5
• 6
• No aplica

otro_vj (2 valores)
-------------------
• No
• Sí

localidad_ori (21 valores)
--------------------------
• Antonio Nariño
• Barrios Unidos
• Bosa
• Candelaria
• Chapinero
• Ciudad Bolivar
• Engativa
• Fontibon
• Kennedy
• Los Martires
• No aplica
• Puente Aranda
• Rafael Uribe Uribe
• San Cristóbal
• Santa Fe
• Suba
• Sumapaz
• Teusaquillo
• Tunjuelito
• Usaquen
• Usme

localidad_des (21 valores)
--------------------------
• Antonio Nariño
• Barrios Unidos
• Bosa
• Candelaria
• Chapinero
• Ciudad Bolivar
• Engativa
• Fontibon
• Kennedy
• Los Martires
• No aplica
• Puente Aranda
• Rafael Uribe Uribe
• San Cristóbal
• Santa Fe
• Suba
• Sumapaz

#### Etapas

In [12]:
print("Dataset: Etapas")
explorar_categorias(etapas)

Dataset: Etapas

nom_mun_hg (21 valores)
-----------------------
• Bogotá
• Bojacá
• Cajicá
• Choachí
• Chía
• Cota
• El Rosal
• Facatativá
• Funza
• Gachancipá
• La Calera
• Madrid
• Mosquera
• Sibaté
• Soacha
• Sopó
• Subachoque
• Tabio
• Tenjo
• Tocancipá
• Zipaquirá

estra_hg (7 valores)
--------------------
• 1
• 2
• 3
• 4
• 5
• 6
• No aplica

modo_principal_vj (11 valores)
------------------------------
• A PIE <15 MIN
• A PIE > 15 MIN
• AUTO
• BICICLETA
• ESPECIAL OCUPADO
• INFORMAL
• MOTO
• OTRO
• TAXI OCUPADO
• TRANSPORTE ESCOLAR
• TRANSPORTE PÚBLICO

pago_viaje (8 valores)
----------------------
• No aplica
• No, evadió el pago
• No, por carencia de recursos
• No, por otra razón
• No, porque no era necesario
• No, porque no había dónde pagar
• No, porque tengo subsidio
• Sí

modalidad_pago_vj (11 valores)
------------------------------
• Adelantó saldo
• Efectivo
• Medio electrónico
• NS/NR
• No aplica
• Otro
• Tarjeta bancarizada
• Tarjeta con beneficio
• Tarjeta de un terce

## Indentificadores y llaves

Esta seccion tiene el fin de buscar y estructurar como se indentifican las tuplas o registros dentro de las diferentes tablas

In [14]:
# se agrupan los diferentes datasets
datasets = {
    "Hogares": hogares,
    "Vehiculos": vehiculos,
    "Personas": personas,
    "Viajes": viajes,
    "Etapas": etapas
}

#### Hogares

In [25]:
explorar_keys("Hogares", hogares, datasets)

=== Hogares ===

Primary Keys:
• cod_hog
	Ejemplos:
		- 20301
		- 6088
		- 13017
    Referenciada en: Vehiculos
• key_hg
	Ejemplos:
		- uuid:e3b750ec-55d0-4b6b-8b58-36eac88be4eb
		- uuid:43fa70a1-a15b-4ea0-aa99-ad9fd6c49189
		- uuid:91ff0f20-dab5-4a43-8668-154399f789b0
    Referenciada en: Vehiculos, Personas, Viajes

Foreign Keys:
• cod_viv -> Vehiculos
• cod_utam_hg -> Vehiculos, Personas, Viajes, Etapas
• zat_hg -> Vehiculos, Personas, Viajes, Etapas
• cod_upl_hg -> Vehiculos, Personas, Viajes, Etapas
• estrato_hg -> Vehiculos


#### Vehiculos

In [26]:
explorar_keys("Vehiculos", vehiculos, datasets)

=== Vehiculos ===

Primary Keys:
• cod_vh
	Ejemplos:
		- 1
		- 2
		- 3
    Referenciada en: Ningún dataset

Foreign Keys:
• cod_viv -> Hogares
• cod_hog -> Hogares
• cod_utam_hg -> Hogares, Personas, Viajes, Etapas
• zat_hg -> Hogares, Personas, Viajes, Etapas
• cod_upl_hg -> Hogares, Personas, Viajes, Etapas
• nom_mun_hg -> Personas, Viajes, Etapas
• estrato_hg -> Hogares
• key_hg -> Hogares, Personas, Viajes


#### Personas

In [27]:
explorar_keys("Personas", personas, datasets)

=== Personas ===

Primary Keys:
• cod_per
	Ejemplos:
		- 1
		- 2
		- 3
    Referenciada en: Ningún dataset
• key_persona
	Ejemplos:
		- uuid:2ad8b488-c706-4c60-9fa3-72c9ea976bd6/accepted/B/person_b[1]
		- uuid:2ad8b488-c706-4c60-9fa3-72c9ea976bd6/accepted/B/person_b[2]
		- uuid:69ed0955-c872-42c8-98b8-120fe0db42d7/accepted/B/person_b[1]
    Referenciada en: Ningún dataset

Foreign Keys:
• cod_hg -> Viajes, Etapas
• nom_mun_hg -> Vehiculos, Viajes, Etapas
• cod_upl_hg -> Hogares, Vehiculos, Viajes, Etapas
• cod_utam_hg -> Hogares, Vehiculos, Viajes, Etapas
• zat_hg -> Hogares, Vehiculos, Viajes, Etapas
• estra_hg -> Viajes, Etapas
• edad -> Viajes, Etapas
• sexo -> Viajes, Etapas
• genero -> Viajes, Etapas
• orien_sexual -> Viajes, Etapas
• identidad_etnica -> Viajes, Etapas
• madre_cab_familia -> Viajes
• max_nivel_edu -> Viajes
• ocupacion_principal -> Viajes
• actividad_economica -> Etapas
• condicion_discapacidad -> Etapas
• key_hg -> Hogares, Vehiculos, Viajes


#### Viajes

In [28]:
explorar_keys("Viajes", viajes, datasets)

=== Viajes ===

Primary Keys:
• cod_vj
	Ejemplos:
		- 1
		- 2
		- 3
    Referenciada en: Etapas
• key_viaje
	Ejemplos:
		- uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/accepted/person_d_e[1]/D/journey[1]
		- uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/accepted/person_d_e[1]/D/journey[2]
		- uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/accepted/person_d_e[2]/D/journey[1]
    Referenciada en: Etapas

Foreign Keys:
• cod_hg -> Personas, Etapas
• nom_mun_hg -> Vehiculos, Personas, Etapas
• cod_upl_hg -> Hogares, Vehiculos, Personas, Etapas
• cod_utam_hg -> Hogares, Vehiculos, Personas, Etapas
• zat_hg -> Hogares, Vehiculos, Personas, Etapas
• estra_hg -> Personas, Etapas
• cod_pers -> Etapas
• t_acceso_min -> Etapas
• t_espera_min -> Etapas
• t_egreso_min -> Etapas
• motivo_viaje -> Etapas
• etapas -> Etapas
• edad -> Personas, Etapas
• sexo -> Personas, Etapas
• genero -> Personas, Etapas
• orien_sexual -> Personas, Etapas
• identidad_etnica -> Personas, Etapas
• madre_cab_familia -> Personas
• m

#### Etapas

In [29]:
explorar_keys("Etapas", etapas, datasets)

=== Etapas ===

Primary Keys:
• cod_eta
	Ejemplos:
		- 1
		- 2
		- 3
    Referenciada en: Ningún dataset
• key_etapa
	Ejemplos:
		- uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/accepted/person_d_e[1]/D/journey[1]/n_journey/stage[1]
		- uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/accepted/person_d_e[1]/D/journey[2]/n_journey/stage[1]
		- uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/accepted/person_d_e[2]/D/journey[1]/n_journey/stage[1]
    Referenciada en: Ningún dataset

Foreign Keys:
• cod_vj -> Viajes
• cod_pers -> Viajes
• cod_hg -> Personas, Viajes
• nom_mun_hg -> Vehiculos, Personas, Viajes
• cod_upl_hg -> Hogares, Vehiculos, Personas, Viajes
• cod_utam_hg -> Hogares, Vehiculos, Personas, Viajes
• zat_hg -> Hogares, Vehiculos, Personas, Viajes
• estra_hg -> Personas, Viajes
• etapas -> Viajes
• t_acceso_min -> Viajes
• t_espera_min -> Viajes
• t_egreso_min -> Viajes
• motivo_viaje -> Viajes
• edad -> Personas, Viajes
• sexo -> Personas, Viajes
• genero -> Personas, Viajes
• orien_sexual -